# Trump Tweet Cleaning and Thread Reconstruction

<p><i>Author:</i> Angelica Vanti</p>

<p><i>Project:</i> Predicting EUR/USD Movements Using Geopolitical and Macroeconomic Variables</p>

<p><i>Notebook:</i> 03 - Trump Tweet Cleaning and Thread Reconstruction</p>

This notebook prepares Donald Trump's first-presidency tweet dataset for subsequent geopolitical-risk annotation using a large language model.

The preprocessing procedure is designed to retain the substantive content and timing of the original tweets while removing observations that are not directly authored by Trump and reconstructing likely multi-part tweet threads.

### Objectives

1. Import and inspect the raw tweet dataset.
2. Retain only the variables required for subsequent analysis.
3. Convert timestamps and sort observations chronologically.
4. Remove retweeted posts and duplicate observations.
5. Clean text encoding and formatting while preserving substantive punctuation and capitalisation.
6. Identify likely continuation tweets using temporal proximity and textual continuation markers.
7. Reconstruct multi-part tweet threads where the continuation criteria are satisfied.
8. Validate the reconstructed threads and export the cleaned tweet dataset for subsequent annotation.

In [1]:
from pathlib import Path
import html
import re
import ftfy
import numpy as np
import pandas as pd

from datetime import timedelta

DATA_PATH = Path(r"C:\Users\franc\OneDrive\Geop-Model\data\trump\twitter2009to2021\LLM_Scores.xlsx")

print(DATA_PATH.exists())

xls = pd.ExcelFile(DATA_PATH)
print(xls.sheet_names)

tweets = pd.read_excel(
    DATA_PATH,
    sheet_name="2017-2021 Tweets",
    dtype={
        "Tweet ID": str,}
)
# Standardise Tweet IDs as strings
tweets["Tweet ID"] = tweets["Tweet ID"].astype(str).str.strip()

# See which IDs are not normal 18–19 digit Twitter IDs
bad_ids = tweets[
    ~tweets["Tweet ID"].str.fullmatch(r"\d{18,19}", na=False)
]

print("Bad IDs:", len(bad_ids))
display(bad_ids[["Tweet ID", "Tweet", "Date"]])

tweets["Tweet ID"] = (
    tweets["Tweet ID"].str.strip().str.replace(r"\.0$", "", regex=True)
)

print(tweets.shape)
tweets.head()


tweets.info()

True
['Sanctions - MSCI Annotations', '2017-2021 Tweets', 'Trade - MSCI Annotations', 'Fed - MSCI Annotations', 'Final LLM Dataset ']
Bad IDs: 5797


,Tweet ID,Tweet,Date
7,815973752785793e3,"Chicago murder rate is record setting - 4,331 ...",2017-01-02T17:31:17Z
16,816298944456233e3,"With all that Congress has to work on, do they...",2017-01-03T15:03:29Z
40,817166353266262e3,The Democratic National Committee would not al...,2017-01-06T00:30:15Z
43,817334794958807e3,Hillary and the Dems were never going to beat ...,2017-01-06T11:39:35Z
53,817701436096127e3,Intelligence stated very strongly there was ab...,2017-01-07T11:56:29Z
...,...,...,...
26357,1346805405998076e3,Sleepy Eyes Chuck Todd is so happy with the fa...,2021-01-06T13:06:45Z
26359,1346809349214249e3,"THE REPUBLICAN PARTY AND, MORE IMPORTANTLY, OU...",2021-01-06T13:22:26Z
26365,1346900434540241e3,Mike Pence didnâ€™t have the courage to do wha...,2021-01-06T19:24:22Z
26368,1346928882595885e3,https://t.co/Pm2PKV0Fp3,2021-01-06T21:17:24Z


(26373, 11)
<class 'pandas.DataFrame'>
RangeIndex: 26373 entries, 0 to 26372
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Tweet ID          26373 non-null  str           
 1   Tweet             26373 non-null  str           
 2   Is Retweeted      26373 non-null  bool          
 3   Is Deleted        26373 non-null  bool          
 4   Device            26373 non-null  str           
 5   Favourites        26373 non-null  int64         
 6   Retweets          26373 non-null  int64         
 7   Date              26373 non-null  str           
 8   Is Flagged        26373 non-null  bool          
 9   Date of Flagging  26373 non-null  datetime64[us]
 10  Unnamed: 10       2 non-null      str           
dtypes: bool(3), datetime64[us](1), int64(2), str(5)
memory usage: 1.7 MB


### Tweet ID Integrity

The source spreadsheet contains a number of Tweet IDs that were previously stored in scientific notation. Because Twitter IDs exceed the precision reliably represented by spreadsheet floating-point formats, some original identifiers cannot be reconstructed exactly from this file.

Tweet IDs are therefore retained as strings and used only as reference identifiers within the cleaning workflow. The substantive modelling variables are based on tweet text and timestamps rather than the numerical value of the Tweet ID.

This limitation is relevant when matching observations across independently exported files and is therefore documented explicitly.

In [2]:
## check that we can see all headers from the file
tweets.columns.tolist()

['Tweet ID',
 'Tweet',
 'Is Retweeted',
 'Is Deleted',
 'Device',
 'Favourites',
 'Retweets',
 'Date',
 'Is Flagged',
 'Date of Flagging',
 'Unnamed: 10']

## 1. Initial Dataset Preparation

The raw tweet dataset is first inspected and reduced to the variables required for the subsequent analysis. Tweet timestamps are converted to a consistent datetime format and observations are ordered chronologically to support the later identification of multi-part tweet sequences.

In [2]:
## We only want to keep the columns necessary for rearranging and extraction.

tweets = tweets[[
    "Tweet ID",
    "Tweet",
    "Date"
]].copy()




In [3]:
## Convert the timestamps for ease

tweets["Date"] = pd.to_datetime(tweets["Date"], utc=True)

tweets = tweets.sort_values("Date").reset_index(drop=True)

## 2. Removal of Retweets and Duplicate Observations

Retweeted posts are removed so that the geopolitical-risk measures are based on content posted directly by Donald Trump's account rather than material originally authored by other users.

Retweets are identified as observations whose text begins with `RT` followed by another account handle. The retweet and favourite-count variables in the original dataset are not used for this purpose, as these record engagement with Trump's own tweets rather than identifying posts that he retweeted.

Duplicate observations are subsequently removed to prevent identical tweets from being represented more than once in the dataset and therefore receiving disproportionate weight in the later geopolitical-risk indices.

In [4]:
## Remove retweets

before = len(tweets)

tweets = tweets[~tweets["Tweet"].str.startswith("RT @", na=False)].copy()

after = len(tweets)

print(f"Removed {before - after} retweeted posts.")

Removed 9507 retweeted posts.


In [5]:
## Remove duplicates
before = len(tweets)

tweets = tweets.drop_duplicates(subset=["Tweet", "Date"]).copy()

after = len(tweets)

print(f"Removed {before - after} duplicate tweets.")

Removed 0 duplicate tweets.


## 3. Tweet Text Cleaning

Following the removal of retweets and duplicate observations, the remaining tweet text is cleaned before thread reconstruction and subsequent LLM annotation.

The cleaning procedure corrects text-encoding artefacts, converts HTML entities to their readable form, and normalises unnecessary whitespace. The procedure is intentionally conservative: substantive wording, punctuation and capitalisation are preserved because these features may provide contextual information for both the identification of multi-part tweet threads and the subsequent geopolitical-risk assessment.

The cleaned text is stored separately from the original tweet text so that the original observations remain available for reference and validation.

In [6]:
## Clean the dataset

def clean_tweet_text(text):
    """
    Clean tweet text while preserving punctuation and capitalisation.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # Convert HTML entities, e.g. &amp; -> &
    text = html.unescape(text)
    text = ftfy.fix_text(text)
    # Fix common encoding problems where possible
    try:
        text = text.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

    # Replace non-breaking spaces with ordinary spaces
    text = text.replace("\xa0", " ")

    # Collapse repeated whitespace, tabs and line breaks
    text = re.sub(r"\s+", " ", text)

    return text.strip()

tweets["Tweet_clean"] = tweets["Tweet"].apply(clean_tweet_text)

changed = (tweets["Tweet"] != tweets["Tweet_clean"]).sum()

print(f"Cleaned the text of {changed} tweets.")



Cleaned the text of 6345 tweets.


In [7]:
tweets["Tweet_clean"] = tweets["Tweet"].apply(clean_tweet_text)

## 4. Identification and Reconstruction of Multi-Part Tweet Threads

Some posts in the source dataset form consecutive parts of a longer statement. Treating these observations independently could remove context that is relevant to the subsequent geopolitical-risk assessment.

Potential thread continuations are identified using two conditions:

1. consecutive tweets must be posted no more than **15 minutes apart**; and
2. the text must contain continuation markers consistent with a split statement.

A tweet is treated as a potential continuation start when it begins with repeated periods or an ellipsis, allowing for leading quotation marks, brackets or whitespace. A preceding tweet is treated as a potential continuation end when it ends with repeated periods, an ellipsis or a question mark.

Tweets are processed in chronological order, with their original dataset ordering retained for observations sharing an identical timestamp. Potential reverse-order continuation pairs at identical timestamps can be identified during candidate inspection; however, the final reconstruction procedure remains conservative and only merges sequences that proceed in the retained chronological ordering.

The procedure is intentionally conservative: uncertain continuations are retained as separate observations rather than forcibly merged.

In [8]:

def starts_with_continuation(text):
    if pd.isna(text):
        return False

    text = str(text).strip()

    return bool(
        re.match(r'^[\'"“”‘’(\[]*\s*(?:\.{2,}|…+)', text)
    )


def ends_with_continuation(text):
    if pd.isna(text):
        return False

    text = str(text).strip()

    return bool(
        re.search(
            r'(?:\.{2,}|…+|\?)[\'"“”‘’)\]]*\s*$',
            text
        )
    )

tweets["Starts_continuation"] = (
    tweets["Tweet_clean"].apply(starts_with_continuation)
)

tweets["Ends_continuation"] = (
    tweets["Tweet_clean"].apply(ends_with_continuation)
)

print(
    f"{tweets['Starts_continuation'].sum()} tweets "
    "start with a continuation marker."
)

print(
    f"{tweets['Ends_continuation'].sum()} tweets "
    "end with a possible continuation marker."
)


1444 tweets start with a continuation marker.
1956 tweets end with a possible continuation marker.


In [9]:
## We sort the tweets in chronological order and then look for tweets that are posted within 15 minutes of each other. If a tweet ends with a continuation marker and the next tweet starts with a continuation marker, we will join them together.

tweets["Original_order"] = range(len(tweets))

tweets = tweets.sort_values(
    ["Date", "Original_order"]
).reset_index(drop=True)


MAX_GAP = pd.Timedelta(minutes=15)


def valid_time_gap(time_a, time_b):
    """
    Return True when two timestamps are no more than 15 minutes apart.
    """

    gap = abs(time_b - time_a)

    return gap <= MAX_GAP

In [10]:
## Main tweet combination loop

def continuation_order(row_a, row_b):
    """
    Determine whether two consecutive tweets form a thread and,
    for identical timestamps, reconstruct their correct order.
    """

    time_a = row_a["Date"]
    time_b = row_b["Date"]

    gap = time_b - time_a

    ## Data should be chronologically sorted
    if gap < pd.Timedelta(0) or gap > MAX_GAP:
        return None

    ## Normal chronological direction
    if (
        row_a["Ends_continuation"]
        and row_b["Starts_continuation"]
    ):
        return row_a, row_b

    ## Reverse only when timestamps are identical
    if (
        time_a == time_b
        and row_b["Ends_continuation"]
        and row_a["Starts_continuation"]
    ):
        return row_b, row_a

    return None

### Candidate Thread-Pair Inspection

Before altering the tweet dataset, all adjacent pairs satisfying the continuation rules are collected for inspection.

Each candidate pair records the two tweet texts, their timestamps, identifiers and the time gap between them. This provides a validation step before the thread-reconstruction algorithm is applied and helps assess whether the rule-based criteria are identifying plausible continuations rather than unrelated tweets posted close together.

In [11]:
## Human check before final alteration

candidate_pairs = []

for i in range(len(tweets) - 1):
    row_a = tweets.iloc[i]
    row_b = tweets.iloc[i + 1]

    ordered_pair = continuation_order(row_a, row_b)

    if ordered_pair is not None:
        first, second = ordered_pair

        candidate_pairs.append({
            "First_ID": first["Tweet ID"],
            "First_tweet": first["Tweet_clean"],
            "Second_ID": second["Tweet ID"],
            "Second_tweet": second["Tweet_clean"],
            "First_time": first["Date"],
            "Second_time": second["Date"],
            "Gap_minutes": abs(
                second["Date"] - first["Date"]
            ).total_seconds() / 60
        })

candidate_pairs = pd.DataFrame(candidate_pairs)

print(
    f"Found {len(candidate_pairs)} possible continuation pairs."
)

pd.set_option("display.max_colwidth", None)

candidate_pairs[[
    "Gap_minutes",
    "First_tweet",
    "Second_tweet"
]].head(20)



Found 1277 possible continuation pairs.


,Gap_minutes,First_tweet,Second_tweet
0,3.600000,"The Democrats, lead by head clown Chuck Schumer, know how bad ObamaCare is and what a mess they are in. Instead of working to fix it, they..","""...do the typical political thing and BLAME. The fact is ObamaCare was a lie from the beginning.""""Keep you doctor, keep your plan!"""" It is...."""
1,5.150000,"""...do the typical political thing and BLAME. The fact is ObamaCare was a lie from the beginning.""""Keep you doctor, keep your plan!"""" It is....""","...time for Republicans &, Democrats to get together and come up with a healthcare plan that really works - much less expensive &, FAR BETTER!"
2,7.583333,Thr coverage about me in the @nytimes and the @washingtonpost gas been so false and angry that the times actually apologized to its.....,"...dwindling subscribers and readers.They got me wrong right from the beginning and still have not changed course, and never will. DISHONEST"
3,3.566667,"The joint statement of former presidential candidates John McCain &, Lindsey Graham is wrong - they are sadly weak on immigration. The two...","...Senators should focus their energies on ISIS, illegal immigration and border security instead of always looking to start World War III."
4,5.866667,Sen. McCain should not be talking about the success or failure of a mission to the media. Only emboldens the enemy! He's been losing so....,"...long he doesn't know how to win anymore, just look at the mess our country is in - bogged down in conflict all over the place. Our hero.."
5,5.816667,"I am reading that the great border WALL will cost more than the government originally thought, but I have not gotten involved in the.....","...design or negotiations yet. When I do, just like with the F-35 FighterJet or the Air Force One Program, price will come WAY DOWN!"
6,4.666667,"Jeff Sessions is an honest man. He did not say anything wrong. He could have stated his response more accurately, but it was clearly not....",...intentional. This whole narrative is a way of saving face for Democrats losing an election that everyone thought they were supposed.....
7,8.066667,...intentional. This whole narrative is a way of saving face for Democrats losing an election that everyone thought they were supposed.....,"...to win. The Democrats are overplaying their hand. They lost the election, and now they have lost their grip on reality. The real story..."
8,3.250000,"...to win. The Democrats are overplaying their hand. They lost the election, and now they have lost their grip on reality. The real story...","""...is all of the illegal leaks of classified and other information. It is a total """"witch hunt!"""""""
9,7.933333,"Despite what you have heard from the FAKE NEWS, I had a GREAT meeting with German Chancellor Angela Merkel. Nevertheless, Germany owes.....","...vast sums of money to NATO &, the United States must be paid more for the powerful, and very expensive, defense it provides to Germany!"


### Thread Reconstruction

The validated continuation rules are now applied sequentially to reconstruct likely multi-part tweet threads.

Starting from each tweet, consecutive observations are merged while they satisfy the temporal and textual continuation criteria. The cleaned tweet texts are concatenated in the inferred order, while the associated Tweet IDs and thread start/end timestamps are retained for traceability.

The procedure stops a thread as soon as the next adjacent observation fails the continuation criteria, preventing unrelated nearby tweets from being incorporated into the same reconstructed statement.

In [12]:
## Merging algorithm

merged_rows = []
i = 0

while i < len(tweets):
    current = tweets.iloc[i]

    combined_text = current["Tweet_clean"]
    combined_ids = [current["Tweet ID"]]
    thread_start = current["Date"]
    thread_end = current["Date"]

    j = i

    while j < len(tweets) - 1:
        row_a = tweets.iloc[j]
        row_b = tweets.iloc[j + 1]

        ordered_pair = continuation_order(row_a, row_b)

        if ordered_pair is None:
            break

        first, second = ordered_pair

        ## For normal chronological chains, the current row must be the first part
        if first["Tweet ID"] != row_a["Tweet ID"]:
            break

        combined_text = (
            combined_text.rstrip()
            + " "
            + second["Tweet_clean"].lstrip()
        )

        combined_ids.append(second["Tweet ID"])
        thread_end = second["Date"]

        j += 1

    merged_rows.append({
        "Tweet ID": combined_ids[0],
        "Tweet IDs": combined_ids,
        "Tweet_clean": combined_text,
        "Date": thread_start,
        "Thread_end": thread_end,
        "Thread_length": len(combined_ids)
    })

    i = j + 1

tweets_merged = pd.DataFrame(merged_rows)

### Thread Reconstruction Diagnostics

The reconstructed dataset is inspected by examining the distribution of thread lengths and reviewing examples containing three or more component tweets.

This diagnostic provides an additional check that the sequential merging procedure has not produced implausibly long or incorrectly constructed threads before the cleaned dataset is exported.

In [ ]:
# Distribution of reconstructed thread lengths
display(
    tweets_merged["Thread_length"]
    .value_counts()
    .sort_index()
    .rename("Number of observations")
    .to_frame()
)

# Inspect examples containing three or more component tweets
display(
    tweets_merged.loc[
        tweets_merged["Thread_length"] >= 3,
        ["Date", "Thread_length", "Tweet IDs", "Tweet_clean"]
    ].head(20)
)

,Date,Thread_length,Tweet IDs,Tweet_clean
28,2017-01-05 11:57:57+00:00,3,"[816977032433270800, 816977937731878900, 816979231217483800]","The Democrats, lead by head clown Chuck Schumer, know how bad ObamaCare is and what a mess they are in. Instead of working to fix it, they.. ""...do the typical political thing and BLAME. The fact is ObamaCare was a lie from the beginning.""""Keep you doctor, keep your plan!"""" It is...."" ...time for Republicans &, Democrats to get together and come up with a healthcare plan that really works - much less expensive &, FAR BETTER!"
358,2017-03-03 02:22:49+00:00,4,"[837488402438176800, 837489578193846300, 837491607171629e3, 837492425283219500]","Jeff Sessions is an honest man. He did not say anything wrong. He could have stated his response more accurately, but it was clearly not.... ...intentional. This whole narrative is a way of saving face for Democrats losing an election that everyone thought they were supposed..... ...to win. The Democrats are overplaying their hand. They lost the election, and now they have lost their grip on reality. The real story... ""...is all of the illegal leaks of classified and other information. It is a total """"witch hunt!"""""""
973,2017-07-09 11:50:24+00:00,3,"[884016887692234800, 884018776391503900, 884020939264073700]","Putin &, I discussed forming an impenetrable Cyber Security unit so that election hacking, &, many other negative things, will be guarded.. ...and safe. Questions were asked about why the CIA &, FBI had to ask the DNC 13 times for their SERVER, and were rejected, still don't.... ...have it. Fake News said 17 intel agencies when actually 4 (had to apologize). Why did Obama do NOTHING when he had info before election?"
1103,2017-07-26 12:55:58+00:00,3,"[890193981585444900, 890196164313833500, 890197095151546400]","After consultation with my Generals and military experts, please be advised that the United States Government will not accept or allow...... ....Transgender individuals to serve in any capacity in the U.S. Military. Our military must be focused on decisive and overwhelming..... ....victory and cannot be burdened with the tremendous medical costs and disruption that transgender in the military would entail. Thank you"
1173,2017-08-07 10:58:09+00:00,3,"[894512983384129500, 894514535062790100, 894515865802223600]","The Trump base is far bigger &, stronger than ever before (despite some phony Fake News polling). Look at rallies in Penn, Iowa, Ohio....... ...and West Virginia. The fact is the Fake News Russian collusion story, record Stock Market, border security, military strength, jobs..... ... Supreme Court pick, economic enthusiasm, deregulation &, so much more have driven the Trump base even closer together. Will never change!"
1240,2017-08-17 13:07:28+00:00,3,"[898169407213645800, 898171544236687400, 898172999945392100]","Sad to see the history and culture of our great country being ripped apart with the removal of our beautiful statues and monuments. You..... ...can't change history, but you can learn from it. Robert E Lee, Stonewall Jackson - who's next, Washington, Jefferson? So foolish! Also... ...the beauty that is being taken out of our cities, towns and parks will be greatly missed and never able to be comparably replaced!"
1506,2017-09-26 00:45:48+00:00,3,"[912478274508423200, 912479500511965200, 912481556127780900]","Texas &, Florida are doing great but Puerto Rico, which was already suffering from broken infrastructure &, massive debt, is in deep trouble.. ...It's old electrical grid, which was in terrible shape, was devastated. Much of the Island was destroyed, with billions of dollars.... ...owed to Wall Street and the banks which, sadly, must be dealt with. Food, water and medical are top priorities - and doing well. #FEMA"
1572,2017-10-01 12:22:14+00:00,3,"[914465475777695700, 914466534365569e3, 914467502251528200]","We have done a great job with the almost impossible situation in Puerto Rico. Outside of the Fake New

## 5. Export of the Cleaned Tweet Dataset

The reconstructed tweet dataset is prepared for export by removing timezone information from the timestamp columns, ensuring compatibility with Excel and subsequent processing notebooks.

The resulting file contains one observation per cleaned standalone tweet or reconstructed tweet thread, together with the associated identifiers and thread timing information.

In [15]:
tweets_export = tweets_merged.copy()

# Remove timezone information
tweets_export["Date"] = tweets_export["Date"].dt.tz_localize(None)
tweets_export["Thread_end"] = tweets_export["Thread_end"].dt.tz_localize(None)


tweets_export.to_excel(
    "C:\\Users\\franc\\OneDrive\\Geop-Model\\data\\processed\\tweets_cleaned_merged_master.xlsx",
    index=False
)

In [16]:
tweets_merged[
    tweets_merged["Tweet ID"].str.contains("e", case=False, na=False)
]

,Tweet ID,Tweet IDs,Tweet_clean,Date,Thread_end,Thread_length
2,815973752785793e3,[815973752785793e3],"Chicago murder rate is record setting - 4,331 shooting victims with 762 murders in 2016. If Mayor can't do it he must ask for Federal help!",2017-01-02 17:31:17+00:00,2017-01-02 17:31:17+00:00,1
11,816298944456233e3,[816298944456233e3],"With all that Congress has to work on, do they really have to make the weakening of the Independent Ethics Watchdog, as unfair as it",2017-01-03 15:03:29+00:00,2017-01-03 15:03:29+00:00,1
33,817166353266262e3,[817166353266262e3],The Democratic National Committee would not allow the FBI to study or see its computer info after it was supposedly hacked by Russia......,2017-01-06 00:30:15+00:00,2017-01-06 00:30:15+00:00,1
36,817334794958807e3,[817334794958807e3],Hillary and the Dems were never going to beat the PASSION of my voters. They saw what was happening in the last two weeks before the......,2017-01-06 11:39:35+00:00,2017-01-06 11:39:35+00:00,1
46,817701436096127e3,[817701436096127e3],Intelligence stated very strongly there was absolutely no evidence that hacking affected the election results. Voting machines not touched!,2017-01-07 11:56:29+00:00,2017-01-07 11:56:29+00:00,1
...,...,...,...,...,...,...
15918,1346805405998076e3,[1346805405998076e3],Sleepy Eyes Chuck Todd is so happy with the fake voter tabulation process that he can't even get the words out straight. Sad to watch!,2021-01-06 13:06:45+00:00,2021-01-06 13:06:45+00:00,1
15920,1346809349214249e3,[1346809349214249e3],"THE REPUBLICAN PARTY AND, MORE IMPORTANTLY, OUR COUNTRY, NEEDS THE PRESIDENCY MORE THAN EVER BEFORE - THE POWER OF THE VETO. STAY STRONG!",2021-01-06 13:22:26+00:00,2021-01-06 13:22:26+00:00,1
15926,1346900434540241e3,[1346900434540241e3],"Mike Pence didn't have the courage to do what should have been done to protect our Country and our Constitution, giving States a chance to certify a corrected set of facts, not the fraudulent or inaccurate ones which they were asked to previously certify. USA demands the truth!",2021-01-06 19:24:22+00:00,2021-01-06 19:24:22+00:00,1
15929,1346928882595885e3,[1346928882595885e3],https://t.co/Pm2PKV0Fp3,2021-01-06 21:17:24+00:00,2021-01-06 21:17:24+00:00,1


## 6. Summary

This notebook prepared Donald Trump's first-presidency tweet dataset for subsequent geopolitical-risk annotation.

The raw dataset was inspected and reduced to the variables required for analysis. Tweet timestamps were converted to a consistent datetime format and observations were ordered chronologically. Retweeted posts and duplicate observations were removed so that the final dataset reflected content posted directly by Trump's account without repeated observations.

Tweet text was then cleaned conservatively to correct encoding and formatting artefacts while preserving substantive language, punctuation and capitalisation.

Potential multi-part tweet threads were identified using a rule-based procedure combining textual continuation markers with a maximum time gap of 15 minutes between adjacent observations. Candidate continuation pairs were generated for manual inspection before the reconstruction algorithm was applied. Likely thread components were then combined into single observations while retaining their constituent Tweet IDs, timestamps and thread lengths for traceability.

The resulting cleaned and reconstructed tweet dataset forms the textual input for the subsequent human annotation and LLM geopolitical-risk scoring stages.

A small number of source Tweet IDs were found to have been stored in scientific notation and could not be reconstructed exactly. As the empirical analysis is based on tweet text and timestamps rather than the numerical values of the Tweet IDs, these identifiers are retained only for reference and this issue is treated as a data-quality limitation.